In [1]:
# pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# pip install transformers==4.32.0 peft==0.5.0
# pip install sentencepiece
# pip install accelerate
# pip install datasets
# pip install bitsandbytes
# pip install gradio
# pip install mdtex2html
# pip install scipy

In [2]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from datasets import load_from_disk
from peft import PeftModel
import re

/home/ava/hkoladara/env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load Models
base_model = "NousResearch/Llama-2-7b-chat-hf"
peft_model = "FinGPT/fingpt-forecaster_dow30_llama2-7b_lora"

In [4]:
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
model = LlamaForCausalLM.from_pretrained(
    base_model, 
    trust_remote_code=True, 
    device_map = "auto", 
    load_in_8bit = True,)
# model = PeftModel.from_pretrained(model, peft_model)
# model = model.eval()


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.06s/it]
/home/ava/hkoladara/env/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/ava/hkoladara/env/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only u

In [ ]:
test_dataset = load_from_disk('fingpt-forecaster-dow30v3-20221231-20230531-llama/')['test']

In [6]:
def test_demo(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt, return_tensors='pt',
        padding=False, max_length=4096
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
        
    res = model.generate(
        **inputs, max_length=4096, do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    output = tokenizer.decode(res[0], skip_special_tokens=True)
    return output    
    # return res

In [ ]:
prompt = """{'role': 'system', 'content': }, {'role': 'user', 'content': }"""
output = test_demo(model, tokenizer, prompt)


In [7]:
base_prompt = "<s>[INST]\n<<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_prompt}[/INST]"
instruction = "You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]:\n...\n"
user_prompt = "[Company Introduction]:\n\nWalgreens Boots Alliance Inc is a leading entity in the Retail sector. Incorporated and publicly traded since 2014-12-31, the company has established its reputation as one of the key players in the market. As of today, Walgreens Boots Alliance Inc has a market capitalization of 22450.74 in USD, with 863.91 shares outstanding.\n\nWalgreens Boots Alliance Inc operates primarily in the US, trading under the ticker WBA on the NASDAQ NMS - GLOBAL MARKET. As a dominant force in the Retail space, the company continues to innovate and drive progress within the industry.\n\nFrom 2020-01-05 to 2020-01-12, WBA's stock price decreased from 48.59 to 44.60. Company news during this period are listed below:\n\nNo relative news reported.\n[Basic Financials]:\n\nNo basic financial reported.\n\nBased on all the information before 2020-01-12, let's first analyze the positive developments and potential concerns for WBA. Come up with 2-4 most important factors respectively and keep them concise. Most factors should be inferred from company related news. Then let's assume your prediction for next week (2020-01-12 to 2020-01-19) is up by 0-1%. Provide a summary analysis to support your prediction. The prediction result need to be inferred from your analysis at the end, and thus not appearing as a foundational factor of your analysis."
print(base_prompt)

<s>[INST]
<<SYS>>
{system_prompt}
<</SYS>>

{user_prompt}[/INST]


In [8]:
input = base_prompt.format(system_prompt = instruction,
                               user_prompt = user_prompt)

In [9]:
output = test_demo(model, tokenizer, input)
print(output)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/home/ava/hkoladara/env/lib/python3.8/site-packages/transformers/generation/utils.py:1411: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


[INST]
<<SYS>>
You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:

[Positive Developments]:
1. ...

[Potential Concerns]:
1. ...

[Prediction & Analysis]:
...

<</SYS>>

[Company Introduction]:

Walgreens Boots Alliance Inc is a leading entity in the Retail sector. Incorporated and publicly traded since 2014-12-31, the company has established its reputation as one of the key players in the market. As of today, Walgreens Boots Alliance Inc has a market capitalization of 22450.74 in USD, with 863.91 shares outstanding.

Walgreens Boots Alliance Inc operates primarily in the US, trading under the ticker WBA on the NASDAQ NMS - GLOBAL MARKET. As a dominant force in the Retail space, the company continues to inno

In [ ]:
As a seasoned stock market analyst, I have analyzed the recent news and financials for Walgreens Boots Alliance Inc (WBA) and identified key positive developments and potential concerns. Based on these factors, I provide a prediction for WBA\'s stock price movement for the upcoming week.\n\nPositive Developments:\n\n1. Strong Global Presence: WBA has a strong presence in over 25 countries, which provides a diverse revenue stream and mitigates risks associated with any single market. This global reach also enables the company to leverage its scale and resources to drive growth in various geographies.\n2. Innovation and E-commerce Strength: WBA has been at the forefront of Retail innovation, particularly in the areas of e-commerce and digital health solutions. The company\'s investments in these areas have positioned it well to capitalize on the growing demand for online shopping and virtual health services.\n3. Cost Management and Efficiency Improvements: WBA has been focusing on cost management and efficiency improvements, which has resulted in better profitability and cash flow generation. The company\'s efforts to optimize its supply chain, streamline operations, and reduce administrative expenses have further enhanced its financial position.\n\nPotential Concerns:\n\n1. Competition and Market Share: The Retail industry is highly competitive, and WBA faces intense competition from other major players such as CVS Health and Rite Aid. As the market continues to consolidate, WBA may experience pressure on its market share, which could impact its financial performance.\n2. Regulatory and Reputation Risks: WBA operates in a highly regulated industry, and any changes in government policies or regulatory frameworks could impact its business operations. Additionally, the company has faced reputation risks in the past due to issues such as data breaches or quality control lapses, which could dampen customer trust and impact its financial performance.\n\nBased on these factors, I predict that WBA\'s stock price will remain relatively stable next week (2020-01-12 to 2020-01-19), with a possible slight increase of up to 0.5%. This prediction is rooted in the company\'s strong global presence, commitment to innovation and e-commerce, and ongoing cost management efforts. However, the impact of competition and regulatory risks cannot be ruled out, and these factors could potentially limit the stock\'s upside potential.\n\nIn summary, while WBA\'s fundamentals remain strong, the company faces challenges from a highly competitive Retail landscape and regulatory risks. Nonetheless, the company\'s investments in innovation and cost management efforts position it well to navigate these challenges and continue to deliver shareholder value in the long term.'

In [10]:
peft_model = PeftModel.from_pretrained(model, peft_model)
peft_model = peft_model.eval()


In [11]:
def test_demo_peft(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt, return_tensors='pt',
        padding=False, max_length=4096
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
        
    res = peft_model.generate(
        **inputs, max_length=4096, do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    output = tokenizer.decode(res[0], skip_special_tokens=True)
    return output    
    # return res

In [13]:
output_peft = test_demo(peft_model, tokenizer, input)
print(output_peft)

[INST]
<<SYS>>
You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:

[Positive Developments]:
1. ...

[Potential Concerns]:
1. ...

[Prediction & Analysis]:
...

<</SYS>>

[Company Introduction]:

Walgreens Boots Alliance Inc is a leading entity in the Retail sector. Incorporated and publicly traded since 2014-12-31, the company has established its reputation as one of the key players in the market. As of today, Walgreens Boots Alliance Inc has a market capitalization of 22450.74 in USD, with 863.91 shares outstanding.

Walgreens Boots Alliance Inc operates primarily in the US, trading under the ticker WBA on the NASDAQ NMS - GLOBAL MARKET. As a dominant force in the Retail space, the company continues to inno

In [15]:
answer = re.sub(r'.*\[/INST\]\s*', '', output_peft, flags=re.DOTALL)
print(answer)

[Positive Developments]:

1. The lack of relative news indicates stability and consistency in the company's operations, which can be interpreted as a positive sign.
2. The company's consistent market capitalization and the number of shares outstanding show the resilience and endurance of its stocks even during a time when its stock price has been slightly down.
3. With no significant news reported recently, this could imply stability in company operations and financials.

[Potential Concerns]:

1. The recent decrease in the stock price should be a cause for concern for investors and analysts, indicating potential instability in the stock's performance.
2. The company's market capitalization and share count have seen a decline which could point towards some underlying concerns with the company's financial health.

[Prediction & Analysis]:

Based on the aforementioned indicators, my prediction for the stock price movement in the upcoming week (2020-01-12 to 2020-01-19) for Walgreens Boot

In [ ]:
answers, gts = [], []

for i in range(len(test_dataset)):
    prompt = test_dataset[i]['prompt']
    output = test_demo(model, tokenizer, prompt)
    answer = re.sub(r'.*\[/INST\]\s*', '', output, flags=re.DOTALL)
    gt = test_dataset[i]['answer']
    print('\n------- Prompt ------\n')
    print(prompt)
    print('\n------- LLaMA2 Finetuned ------\n')
    print(answer)
    print('\n------- GPT4 Groundtruth ------\n')
    print(gt)
    print('\n===============\n')
    answers.append(answer)
    gts.append(gt)

In [ ]:
calc_metrics(answers, gts)


In [ ]:
!unzip "2023-10-01_2023-12-18.zip"